Пример скрипта


In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, IntegerType, DoubleType, BooleanType
import traceback
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


In [17]:
def create_spark_session(s3_config=None):
    """
    Create and configure a Spark session.

    Parameters
    ----------
    s3_config : dict, optional
        Dictionary containing S3 configuration parameters
        (endpoint_url, access_key, secret_key)

    Returns
    -------
    SparkSession
        Configured Spark session
    """
    print("DEBUG: Начинаем создание Spark сессии")
    try:
        # Создаем базовый Builder
        builder = (SparkSession
            .builder
            .appName("FraudDetectionModel")
        )

        # Если передана конфигурация S3, добавляем настройки
        if s3_config and all(k in s3_config for k in ['endpoint_url', 'access_key', 'secret_key']):
            print(f"DEBUG: Настраиваем S3 с endpoint_url: {s3_config['endpoint_url']}")
            builder = (builder
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
                .config("spark.hadoop.fs.s3a.endpoint", s3_config['endpoint_url'])
                .config("spark.hadoop.fs.s3a.access.key", s3_config['access_key'])
                .config("spark.hadoop.fs.s3a.secret.key", s3_config['secret_key'])
                .config("spark.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
            )

        print("DEBUG: Spark сессия успешно сконфигурирована")
        # Создаем и возвращаем сессию Spark
        return builder
    except Exception as e:
        print(f"ERROR: Ошибка создания Spark сессии: {str(e)}")
        print(f"Traceback: {traceback.format_exc()}")

In [18]:
def clean_convert(spark, source_path: str, output_path: str) -> None:

    # Define the schema
    schema = StructType([
        StructField("transaction_id", LongType(), False),
        StructField("tx_datetime", TimestampType(), False),
        StructField("customer_id", IntegerType(), False),
        StructField("terminal_id", IntegerType(), False),
        StructField("tx_amount", DoubleType(), False),
        StructField("tx_time_seconds", LongType(), False),
        StructField("tx_time_days", IntegerType(), False),
        StructField("tx_fraud", IntegerType(), False),
        StructField("tx_fraud_scenario", IntegerType(), False)
    ])


    print("Try to read CSV files from source bucket...")

    #Read the TXT file as a CSV file
    df_txt = spark.read.csv(
        source_path,
        header=False,
        comment="#",  # comment character
        schema=schema,
        sep=",",  # separator (comma in this case)
        mode="PERMISSIVE"  # Handles lines with more or fewer columns.
    )
    df = df_txt


    # Clean the DataFrame by:
    # 1. Dropping rows where any columns have null values.
    # 2. Removing duplicate rows.
    # 3. Filtering rows to include only those with a positive `tx_amount`.
    df_cleaned = df.na.drop(how="any").distinct().filter(df.tx_amount > 0)

    # Save the cleaned DataFrame as a Parquet file
    df_cleaned.repartition(10).write.mode("overwrite").parquet(output_path)

    # print('Records count after clean:', df_cleaned.count())
    #
    # # Stop the Spark session
    # spark.stop()
    print("Successfully saved the result to the output parquet file")


In [20]:
def read_parquet_and_split(spark, parquet_path: str, train_ratio: float = 0.8):
    """
    Reads a Parquet file, splits it into train and test DataFrames.

    Args:
        spark: The SparkSession.
        parquet_path: The path to the Parquet file.
        train_ratio: The ratio of data to use for training (default: 0.8).

    Returns:
        A tuple containing the train and test DataFrames.
    """
    try:
        df = spark.read.parquet(parquet_path)

        # Split the DataFrame into train and test sets
        train_df, test_df = df.randomSplit([train_ratio, 1 - train_ratio], seed=42)
        print(f"Training set size: {train_df.count()}")
        print(f"Testing set size: {test_df.count()}")
        return train_df, test_df

    except Exception as e:
        print(f"Error reading and splitting Parquet file: {e}")
        traceback.print_exc()
        return None, None

In [21]:
def prepare_features(train_df, test_df):
    """
    Prepare features for model training.

    Parameters
    ----------
    train_df : DataFrame
        Training DataFrame
    test_df : DataFrame
        Testing DataFrame

    Returns
    -------
    tuple
        (train_df, test_df, feature_cols) - Prepared DataFrames and feature column names
    """
    print("DEBUG: Начинаем подготовку признаков")
    try:
        # Получаем типы столбцов
        print("DEBUG: Проверяем типы столбцов")
        dtypes = dict(train_df.dtypes)
        print(f"DEBUG: Типы данных: {dtypes}")

        # Исключаем строковые столбцы и целевую переменную 'fraud'
        # feature_cols = [col for col in train_df.columns
        #                 if col != 'tx_fraud' and col != 'tx_fraud_scenario' ]
        feature_cols = ['customer_id', 'terminal_id', 'tx_amount']
        print(f"DEBUG: Выбрано {len(feature_cols)} числовых признаков: {feature_cols}")

        # Проверим наличие нулевых значений
        print("DEBUG: Проверка наличия нулевых значений в обучающей выборке")
        for col in train_df.columns:
            null_count = train_df.filter(train_df[col].isNull()).count()
            if null_count > 0:
                print(f"WARNING: Колонка '{col}' содержит {null_count} нулевых значений")

        return train_df, test_df, feature_cols
    except Exception as e:
        print(f"ERROR: Ошибка подготовки признаков: {str(e)}")
        print(f"Traceback: {traceback.format_exc()}")
        raise

In [31]:

def train_model(train_df, test_df, feature_cols, model_type="rf"):
    """
    Train a fraud detection model (simplified version without MLflow and hyperparameter tuning)

    Parameters
    ----------
    train_df : DataFrame
        Training DataFrame
    test_df : DataFrame
        Testing DataFrame
    feature_cols : list
        List of feature column names
    model_type : str
        Model type to train ('rf' for Random Forest, 'lr' for Logistic Regression)

    Returns
    -------
    tuple
        (trained_model, metrics) - Trained model and its performance metrics
    """
    print(f"DEBUG: Начинаем обучение модели типа {model_type}")

    try:
        # Create feature vector
        print("DEBUG: Создание преобразователя признаков")
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
        scaler = StandardScaler(
            inputCol="features_raw",
            outputCol="features",
            withStd=True,
            withMean=True
        )

        # Select model based on type
        print("DEBUG: Создание классификатора")
        if model_type == "rf":
            classifier = RandomForestClassifier(
                labelCol="tx_fraud",
                featuresCol="features",
                numTrees=10,  # Фиксированное значение вместо перебора
                maxDepth=5    # Фиксированное значение вместо перебора
            )
        else:
            raise ValueError(f"Unsupported model type: {model_type}")

        # Create pipeline
        print("DEBUG: Создание пайплайна")
        pipeline = Pipeline(stages=[assembler, scaler, classifier])

        # Create evaluators
        print("DEBUG: Создание оценщиков")
        evaluator_auc = BinaryClassificationEvaluator(
            labelCol="tx_fraud",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        )
        evaluator_acc = MulticlassClassificationEvaluator(
            labelCol="tx_fraud",
            predictionCol="prediction",
            metricName="accuracy"
        )
        evaluator_f1 = MulticlassClassificationEvaluator(
            labelCol="tx_fraud",
            predictionCol="prediction",
            metricName="f1"
        )

        # Train the model
        print("DEBUG: Обучаем модель...")
        trained_model = pipeline.fit(train_df)
        print("DEBUG: Модель успешно обучена")

        # Make predictions on test data
        print("DEBUG: Делаем предсказания на тестовых данных")
        predictions = trained_model.transform(test_df)
        print("DEBUG: Предсказания получены")

        # Calculate metrics
        print("DEBUG: Рассчитываем метрики")
        auc = evaluator_auc.evaluate(predictions)
        accuracy = evaluator_acc.evaluate(predictions)
        f1 = evaluator_f1.evaluate(predictions)

        # Print metrics
        print(f"AUC: {auc}")
        print(f"Accuracy: {accuracy}")
        print(f"F1 Score: {f1}")

        metrics = {
            "auc": auc,
            "accuracy": accuracy,
            "f1": f1
        }

        return trained_model, metrics

    except Exception as e:
        print(f"ERROR: Ошибка обучения модели: {str(e)}")
        raise


In [23]:
spark_session = create_spark_session().getOrCreate()


DEBUG: Начинаем создание Spark сессии
DEBUG: Spark сессия успешно сконфигурирована


In [24]:
file_path = "data/2022-09-05.txt"
output_path = "data/2022-09-05.parquet"
clean_convert(spark_session, file_path, output_path)

Try to read CSV files from source bucket...
Successfully saved the result to the output bucket!


In [25]:
# Read the Parquet file and split into train and test sets
train_df, test_df = read_parquet_and_split(spark_session, output_path)
train_df, test_df, feature_cols = prepare_features(train_df, test_df)

Training set size: 37594362
Testing set size: 9398099
DEBUG: Начинаем подготовку признаков
DEBUG: Проверяем типы столбцов
DEBUG: Типы данных: {'transaction_id': 'bigint', 'tx_datetime': 'timestamp', 'customer_id': 'int', 'terminal_id': 'int', 'tx_amount': 'double', 'tx_time_seconds': 'bigint', 'tx_time_days': 'int', 'tx_fraud': 'int', 'tx_fraud_scenario': 'int'}
DEBUG: Выбрано 3 числовых признаков: ['customer_id', 'terminal_id', 'tx_amount']
DEBUG: Проверка наличия нулевых значений в обучающей выборке


In [32]:
model, metrics = train_model(train_df, test_df, feature_cols)

DEBUG: Начинаем обучение модели типа rf
DEBUG: Создание преобразователя признаков
DEBUG: Создание классификатора
DEBUG: Создание пайплайна
DEBUG: Создание оценщиков
DEBUG: Обучаем модель...
DEBUG: Модель успешно обучена
DEBUG: Делаем предсказания на тестовых данных
DEBUG: Предсказания получены
DEBUG: Рассчитываем метрики
AUC: 0.5
Accuracy: 0.945993971759608
F1 Score: 0.9197403564373087


In [33]:
spark_session.stop() #Only stop when you are done with the spark session.